# 파트 1. 벡터 DB와 RAG의 구조

2일차 파트 3에서는 문서 253건의 임베딩을 만들어 질문과 가까운 문서를 찾았습니다. 그 벡터 인덱스는 프로그램을 끝내면 사라지고, 문서가 바뀌면 처음부터 다시 만들어야 했습니다. 실제 사내 문서는 매주 개정되고, 새로 올라오고, 폐기됩니다. 이 파트에서는 인덱스를 파일로 저장하고 바뀐 문서의 행만 고치는 코드를 작성합니다. 그리고 RAG 전체에서 이 인덱스가 어느 단계에 해당하는지 정리합니다. 임베딩 모델을 고르는 기준과, 과제가 RAG에 맞는지 판단하는 기준도 다룹니다.

이 파트를 마치면 다음을 할 수 있습니다.

1. RAG를 인덱싱, 검색, 생성의 세 단계로 나누고, 각 단계가 언제 실행되는지와 응답이 틀렸을 때 어느 단계부터 확인하는지 설명합니다.
2. 벡터 인덱스를 파일로 저장하고 불러오며, 문서 번호와 임베딩 모델의 이름을 함께 저장하는 이유를 설명합니다.
3. 문서의 개정, 추가, 폐기를 바뀐 행만 고쳐 반영하고, 벡터와 문서 번호의 순서가 어긋나면 어떤 일이 생기는지 설명합니다.
4. 임베딩 모델을 고르는 기준(언어, 최대 입력 길이, 차원, 실행 위치)을 말하고, 후보 모델을 같은 질문 세트로 측정해 비교합니다.
5. 임베딩 모델을 바꾸면 인덱스 전체를 다시 만들어야 하는 이유를 실행 결과로 설명합니다.
6. 과제가 RAG에 맞는지 문서의 분량, 문서가 바뀌는 빈도, 출처의 필요 여부로 판단합니다.

예제의 실행 결과는 2026-11-26에 실행한 값이고, 학습 노트북을 실행한 출력을 그대로 실었습니다. 검색 점수와 순위는 같은 임베딩 모델을 쓰면 거의 같은 값이 나옵니다. 이 파트의 예제는 생성 모델 API를 호출하지 않고 임베딩 모델만 PC에서 실행합니다. 노트북의 "준비 (2)" 셀 끝에 있는 연결 확인만 API를 한 번 부릅니다. API를 설정하지 않아 이 확인이 실패해도 앞에서 만든 이름은 그대로 쓸 수 있으므로 예제를 계속 실행해도 됩니다.

## 이 파트에서 만드는 것

> **요점**
>
> 2일차의 벡터 인덱스를 파일로 저장하고, 문서가 바뀔 때 바뀐 행만 고치는 벡터 DB(벡터를 저장하고, 고치고, 검색하는 시스템)를 만듭니다. 문서 변경 목록을 넣으면 코드가 바뀐 행만 고치고, 그 결과 검색 결과가 바뀝니다.

재료는 2일차와 같은 가상 공장 문서 253건과, 이번 주에 문서함에서 바뀐 내용입니다. 변경은 아홉 건이고 세 종류입니다.

| 종류 | 문서 | 내용 |
| --- | --- | --- |
| 개정 | `SOP-ETCH-005` | 시즈닝 런 더미 웨이퍼 25장 → 15장 |
| 개정 | `EHS-001` | 가스 감지기 15분 모니터링 절 추가 |
| 신규 | `NOTI-101` | ETCH-03 정기 PM 요일 화요일 → 목요일 |
| 신규 | `SOP-ETCH-020` | 에치 챔버 웻 클리닝 표준작업지침 |
| 폐기 | `NOTI-001`, `007`, `013`, `019`, `025` | 같은 연말 정기 셧다운 일정 공지가 다섯 번 올라와 있었음. 지난 일정이므로 폐기함 |

개정본과 신규 문서 4건은 `data/3일차/변경_문서/`에 있습니다. 변경 목록은 `data/3일차/변경_목록.csv`입니다.

이 파트에서 만드는 것은 위 변경을 반영하는 코드입니다. 반영하기 전과 후에 같은 질문을 검색하면 1위 문서가 바뀝니다. 다음 표의 결과는 실습 1에서 얻습니다. 예제 1-2는 같은 방법을 문서 한두 건에 먼저 적용합니다.

| 질문 | 반영 전 1위 | 반영 후 1위 |
| --- | --- | --- |
| ETCH-03 PM 요일이 바뀐다던데 무슨 요일이에요? | `MTG-2025-W04` (신규 문서가 아직 없음) | `NOTI-101` |
| 가스 실린더 바꾸고 감지기는 몇 분 동안 봐야 하나요? | `EHS-016` (정답 `EHS-001`은 4위) | `EHS-001` |

질문 2의 반영 전 상위 4건(`EHS-016`, `EHS-006`, `EHS-011`, `EHS-001`)은 점수가 0.885~0.886으로 거의 같아서, PC에 따라 이 네 건의 순서가 바뀔 수 있습니다. 반영한 뒤에는 `EHS-001`이 1위입니다.

이 파트는 다음 순서로 진행합니다.

1. RAG를 인덱싱, 검색, 생성의 세 단계로 나눕니다.
2. 벡터 인덱스를 파일로 저장하고 불러옵니다.
3. 문서의 개정, 추가, 폐기를 바뀐 행만 고쳐 반영합니다.
4. 임베딩 모델을 고르는 기준을 세우고, 모델을 바꾸면 인덱스를 다시 만들어야 하는 이유를 확인합니다.
5. 과제가 RAG에 맞는지 판단하는 기준을 세웁니다.

## 준비

이 노트북은 `노트북` 폴더에서 실행합니다. 같은 폴더의 `llm.py`가 API 호출을 맡습니다. Python 3.10 이상이 필요하고, 패키지는 다음 명령으로 설치합니다.

```bash
python -m pip install anthropic openai pandas pydantic openpyxl jupyter
```

일자마다 더 필요한 패키지가 있으면 아래 "준비 (2)" 셀 안의 `# 설치:` 주석에 명령이 있습니다. 그 명령도 실행합니다.

### API 키는 코드에 적지 않습니다

API 키는 API를 호출한 사람을 식별하는 문자열입니다. 사용 기록과 요금은 키별로 집계됩니다. 키를 가진 사람은 누구나 그 키로 호출할 수 있고, 요금은 키의 주인에게 청구됩니다.

코드에 적은 키는 파일을 공유하거나 저장소에 올리는 순간 함께 퍼집니다. 그래서 키는 환경 변수에 두고 코드에서는 환경 변수를 읽습니다.

> **주의**
>
> 키를 메신저, 화면 공유, 캡처 이미지에 노출하지 않습니다. 노출됐다면 바로 폐기하고 새로 발급받습니다.

### 환경 변수 설정

`llm.py`는 아래 환경 변수를 읽어 어느 API로 보낼지 고릅니다.

**Anthropic**

| 환경 변수 | 값 |
| --- | --- |
| `ANTHROPIC_API_KEY` | 발급받은 키 |

모델 이름은 설정하지 않아도 됩니다. 소형, 기본, 대형 모델 등급에 현행 Claude 모델이 기본값으로 들어 있습니다.

**사내 API(OpenAI 호환)**

| 환경 변수 | 값 |
| --- | --- |
| `LLM_PROVIDER` | `openai` |
| `OPENAI_BASE_URL` | 사내 API 주소. 보통 `/v1`로 끝납니다 |
| `OPENAI_API_KEY` | 사내 API 키 |
| `LLM_MODEL_SMALL` | 소형 모델 이름 |
| `LLM_MODEL_DEFAULT` | 기본 모델 이름 |
| `LLM_MODEL_LARGE` | 대형 모델 이름 |
| `LLM_STRUCTURED` | 구조화 출력을 요청하는 방식. 설정하지 않으면 `json_schema`입니다. 서버가 이 방식을 지원하지 않을 때만 바꿉니다(2일차 파트 2의 "구조화 출력" 절) |

사내에서 쓸 수 있는 모델이 하나뿐이면 세 변수에 같은 이름을 넣습니다. 그러면 모델 등급을 비교하는 예제와 실습에서는 차이가 나지 않습니다.

환경 변수는 Jupyter를 시작하기 전에 설정합니다. `setx`로 설정한 값은 그 뒤에 새로 연 창에만 적용됩니다. Windows PowerShell에서 다음과 같이 설정한 뒤 터미널을 새로 열고, 그 터미널에서 Jupyter를 시작합니다. VS Code로 노트북을 연다면 VS Code 창을 모두 닫았다가 다시 엽니다.

Anthropic API를 쓸 때:

```powershell
setx ANTHROPIC_API_KEY "발급받은_키"
```

사내 API를 쓸 때(위 표의 나머지 변수도 같은 방법으로 설정합니다):

```powershell
setx LLM_PROVIDER openai
setx OPENAI_BASE_URL "사내_API_주소"
setx OPENAI_API_KEY "사내_API_키"
```

`setx`로 설정해 두면 모든 일자의 노트북에서 그대로 씁니다.

환경 변수를 미리 설정하지 못했다면 아래 "준비 (1)" 셀이 키, 주소, 모델 이름을 물어봅니다. 입력한 값은 화면에 보이지 않고 노트북 파일에도 저장되지 않습니다. 이 값은 커널이 살아 있는 동안만 쓰이므로, 노트북을 새로 열거나 커널을 다시 시작하면 다시 입력합니다. 단, 사내 API를 쓰려면 `LLM_PROVIDER=openai`만은 Jupyter를 시작하기 전에 설정해야 합니다. "준비 (1)" 셀이 이 값을 보고 어떤 값을 물을지 정하기 때문입니다. 이 값이 없으면 `llm.py`는 Anthropic API로 요청을 보냅니다.

### 연결 확인

아래 두 셀을 차례로 실행합니다. 두 셀이 오류 없이 끝나고 마지막 줄에 모델의 응답(보통 `준비 완료`)이 나오면 연결된 것입니다. 응답 문구는 모델에 따라 조금 다를 수 있습니다.

In [ ]:
# 준비 (1) 환경 변수가 비어 있으면 여기서 입력합니다
import os
from getpass import getpass

import llm

needed = (['ANTHROPIC_API_KEY'] if llm.provider() == 'anthropic'
          else ['OPENAI_BASE_URL', 'OPENAI_API_KEY', 'LLM_MODEL_SMALL', 'LLM_MODEL_DEFAULT',
                'LLM_MODEL_LARGE'])
for name in needed:
    if not os.environ.get(name):
        os.environ[name] = getpass(f'{name}: ').strip()
print('사용하는 API:', llm.provider())

In [ ]:
# 준비 (2) 공통으로 쓰는 것을 불러오고 연결을 확인합니다
import json
import time
from concurrent.futures import ThreadPoolExecutor

import pandas as pd

# 설치: python -m pip install sentence-transformers scikit-learn pypdf python-docx openpyxl pandas
import numpy as np

import embed

DATA = 'data/3일차'
CORPUS = 'data/2일차/corpus'      # 2일차와 같은 문서 253건
OUT = 'out'                        # 예제가 만드는 파일을 두는 폴더
os.makedirs(OUT, exist_ok=True)


for alias in ['소형', '기본', '대형']:
    try:
        print(f'{alias} 모델:', llm.model_name(alias))
    except RuntimeError:
        print(f'{alias} 모델: 정하지 않음')

reply = llm.send([{'role': 'user', 'content': '"준비 완료"라고만 답해 주세요.'}], model='소형', max_tokens=200)
print(reply.text)

오류가 나면 아래 표에서 원인을 찾습니다.

| 나온 문구 | 원인과 조치 |
| --- | --- |
| `환경 변수 ...의 값이 비어 있습니다` | 환경 변수를 설정하지 않았습니다. 변수 이름의 철자를 확인하고 "준비 (1)" 셀을 다시 실행합니다 |
| `환경 변수 LLM_MODEL_...에 '소형' 모델 이름을 넣어 주세요` | 사내 API를 쓰는데 모델 이름을 비워 두었습니다. 커널을 다시 시작하고 "준비 (1)" 셀에서 모델 이름을 입력합니다 |
| `401 auth` | 키가 틀렸습니다. "준비 (1)" 셀은 값이 비어 있을 때만 묻습니다. 셀에서 입력했다면 커널을 다시 시작한 뒤 "준비 (1)" 셀에서 앞뒤 공백이나 따옴표 없이 다시 입력합니다. `setx`로 설정했다면 값을 다시 설정하고 Jupyter를 다시 시작합니다 |
| `403 permission` | 키에 그 모델이나 기능을 쓸 권한이 없습니다. 키를 발급한 곳에 권한을 확인합니다 |
| `404 not_found` | 모델 이름이나 API 주소가 틀렸습니다. 고치는 방법은 `401 auth`와 같습니다 |
| `connection` | 서버에 연결하지 못했습니다. 주소와 사내망 연결을 확인합니다 |
| `timeout` | 정해진 시간 안에 응답이 오지 않았습니다. 잠시 뒤 다시 실행합니다 |
| `NameError` | 앞 셀을 건너뛰었거나 커널을 다시 시작했습니다. 위에서부터 차례로 다시 실행합니다 |

## RAG의 세 단계

> **요점**
>
> RAG는 인덱싱, 검색, 생성의 세 단계입니다. 인덱싱은 문서가 바뀔 때만 실행하고, 검색과 생성은 질문이 올 때마다 실행합니다. 응답이 틀리면 검색 결과부터 확인합니다.

2일차에서 RAG를 검색과 생성의 두 단계로 설명했습니다. 이 두 단계는 질문이 올 때마다 실행됩니다. 질문이 오기 전에 문서를 검색할 수 있게 준비해 두는 일이 하나 더 있고, 이를 인덱싱이라고 합니다. 인덱싱은 문서를 처리해 임베딩으로 바꾸고 벡터 인덱스에 넣는 일입니다. 문서 처리는 원본 문서에서 글을 뽑아 검색 단위로 자르는 일이며 파트 2에서 다룹니다. 인덱싱은 문서가 바뀔 때만 다시 실행합니다.

검색은 질문의 임베딩과 가까운 문서를 점수 순으로 찾습니다. 검색 결과에서 점수가 높은 k건을 top-k라고 합니다. k는 정수이고, RAG에서는 프롬프트에 넣을 문서의 건수입니다. 아래 그림과 표의 "상위 k건"이 top-k입니다.

<img alt="RAG의 세 단계. 위쪽 줄은 문서가 바뀔 때만 실행하는 인덱싱으로, 원본 문서를 문서 처리와 임베딩을 거쳐 벡터 DB에 저장한다. 아래쪽 줄은 질문이 올 때마다 실행하는 검색과 생성으로, 질문과 가까운 상위 k건을 벡터 DB에서 찾고, 찾은 문서와 질문을 모델에 보내 응답과 출처를 받는다." width="680" src="">

*위쪽 줄은 문서가 바뀔 때만 실행하는 인덱싱이고, 아래쪽 줄은 질문이 올 때마다 실행하는 검색과 생성입니다. 청록색 점선은 인덱싱이 만든 벡터 DB를 검색 상자가 계속 쓴다는 뜻이고, 주황색 상자는 모델을 부르는 생성 단계입니다. 이 교재는 파트 1에서 벡터 DB를, 파트 2에서 문서 처리를, 파트 3에서 생성을 만듭니다.*

세 단계를 비교하면 다음과 같습니다.

| 단계 | 실행 시점 | 받는 것 | 내보내는 것 |
| --- | --- | --- | --- |
| 인덱싱 | 문서가 바뀔 때 | 원본 문서 | 벡터 인덱스(문서의 임베딩) |
| 검색 | 질문마다 | 질문 한 건 | 점수가 높은 순으로 정렬한 상위 k건 |
| 생성 | 질문마다 | 질문과 검색 결과 | 응답과 출처 문서 번호 |

RAG를 세 단계로 나누는 이유는 틀린 응답을 고칠 곳을 찾기 위해서입니다. 응답 하나만 보면 어느 단계가 틀렸는지 알 수 없습니다. 단계를 나누면 다음과 같이 고칠 곳을 정할 수 있습니다.

- 문서가 바뀌었는데 응답이 옛 내용이면 인덱싱을 확인합니다.
- 검색 결과에 질문과 관계없는 문서가 있으면 검색을 고칩니다.
- 검색 결과는 맞는데 응답이 그 내용과 다르거나 형식이 틀리면 생성(프롬프트)을 고칩니다.

검색이 가져오지 못한 내용은 모델이 답할 수 없으므로, 응답이 이상하면 검색 결과부터 확인합니다.

이 교재는 검색을 먼저 만들고 문서 처리를 나중에 다룹니다. 실제 파이프라인에서는 문서 처리가 가장 앞입니다. 검색을 먼저 만들어 보면 문서를 어떻게 다듬어야 하는지가 검색 결과로 드러납니다.

## 벡터 DB와 벡터 인덱스

> **요점**
>
> 벡터 인덱스를 파일로 저장해 두면 문서를 다시 임베딩하지 않아도 됩니다. 인덱스에는 벡터와 함께 문서 번호와 임베딩 모델의 이름을 저장합니다.

벡터 DB는 벡터 인덱스를 저장하고, 바뀐 문서의 행을 고치고, 질문과 가까운 벡터를 찾아 주는 시스템입니다. 서버로 운영하는 제품도 있고, 프로그램 안에서 쓰는 라이브러리도 있습니다. 이 교재에서는 numpy 배열 하나와 문서 번호 목록으로 같은 기능을 직접 구현합니다. 저장, 행 단위 갱신, 유사도 검색이라는 기능은 제품과 같고, 코드가 짧아서 안에서 무슨 일이 일어나는지 볼 수 있습니다. 다만 많은 제품은 검색을 빠르게 하려고 근사 검색(이 절의 끝에서 설명합니다)을 씁니다.

2일차에서는 `embed.passages()`로 만든 벡터 인덱스가 프로그램이 끝나면 사라졌습니다. 임베딩은 문서 수에 비례해 시간이 걸립니다. GPU가 없는 PC에서는 문서 253건의 임베딩에 수십 초가 걸리기도 합니다(예제 1-1이 걸린 시간을 출력합니다. 값은 PC와 실행할 때마다 다릅니다). 문서가 바뀌지 않았다면 다시 계산할 이유가 없으므로 한 번 만든 인덱스를 파일로 저장합니다.

**예제 1-1. 벡터 인덱스를 만들어 저장하고 불러오기**

In [ ]:
# 예제 1-1
import time

names = sorted(file[:-3] for file in os.listdir(CORPUS) if file.endswith('.md'))   # 문서 번호
docs = [open(f'{CORPUS}/{name}.md', encoding='utf-8').read() for name in names]    # names와 같은 순서의 본문

embed.load()                                      # 모델을 먼저 불러와, 아래 시간에 불러오는 시간이 들어가지 않게 합니다
start = time.perf_counter()
vectors = embed.passages(docs)
seconds = time.perf_counter() - start
print(f'{len(docs)}건, 인덱스 {vectors.shape}, {round(vectors.nbytes / 1e6, 3)}MB, 임베딩에 {round(seconds)}초')

임베딩 모델: intfloat/multilingual-e5-small (PC에 저장된 파일을 씁니다)
253건, 인덱스 (253, 384), 0.389MB, 임베딩에 24초


인덱스를 파일로 저장합니다. 벡터는 `index.npy`에, 문서 번호와 본문과 임베딩 모델의 이름은 `index.json`에 저장합니다. 본문도 저장하는 이유는, 검색으로 찾은 문서의 글을 프롬프트에 넣으려면 번호만으로는 부족하기 때문입니다.

In [ ]:
# 예제 1-1 (파일로 저장합니다)
np.save(f'{OUT}/index.npy', vectors)
with open(f'{OUT}/index.json', 'w', encoding='utf-8') as file:
    json.dump({'model': embed.model_name(), 'names': names, 'docs': docs}, file, ensure_ascii=False)

print('저장한 파일:', sorted(os.listdir(OUT)))
print(f"index.npy {os.path.getsize(f'{OUT}/index.npy') // 1024:,}KB")

저장한 파일: ['index.json', 'index.npy']
index.npy 379KB


저장한 파일을 불러와 같은 인덱스인지 확인하고, 2일차의 질문 20개로 검색이 되는지 봅니다. 문서를 다시 임베딩하지 않으므로 질문의 임베딩만 계산합니다.

In [ ]:
# 예제 1-1 (저장한 파일을 불러옵니다)
loaded = np.load(f'{OUT}/index.npy')
with open(f'{OUT}/index.json', encoding='utf-8') as file:
    saved = json.load(file)
print(loaded.shape, len(saved['names']), f'같은 벡터인가: {bool(np.array_equal(loaded, vectors))}', sep=', ')
print('인덱스를 만든 모델:', saved['model'])

questions = pd.read_csv('data/2일차/검색_질문_20.csv')
ranks = []
for question, gold in zip(questions['질문'], questions['정답']):
    scores = loaded @ embed.query(question)
    order = list(np.argsort(-scores, kind='stable'))
    ranks.append(order.index(saved['names'].index(gold)) + 1)
print(f'recall@1 {sum(rank <= 1 for rank in ranks) / len(ranks):.0%}, recall@3 {sum(rank <= 3 for rank in ranks) / len(ranks):.0%}')

(253, 384), 253, 같은 벡터인가: True
인덱스를 만든 모델: intfloat/multilingual-e5-small
recall@1 85%, recall@3 90%


문서가 많아지면 검색 시간이 얼마나 늘어나는지 측정합니다. 실제 문서 대신 384차원 무작위 벡터 10만 개를 만들고, 질문 하나를 모든 벡터와 비교하는 시간을 측정합니다.

In [ ]:
# 예제 1-1 (벡터가 10만 개이면 검색 시간)
rng = np.random.default_rng(0)
many = rng.standard_normal((100_000, 384)).astype(np.float32)   # 검색 시간만 측정하므로 무작위 벡터를 씁니다
many /= np.linalg.norm(many, axis=1, keepdims=True)
repeat = 20                                                        # 한 번만 측정하면 다른 작업의 영향을 받기 쉬워 20번의 평균을 냅니다
start = time.perf_counter()
for _ in range(repeat):
    top3 = np.argsort(-(many @ many[0]), kind='stable')[:3]        # 첫 번째 벡터를 질문으로 삼습니다
milliseconds = (time.perf_counter() - start) / repeat * 1000
print(f'벡터 {len(many):,}개, {many.nbytes / 1e6:.0f}MB, 질문 하나의 검색에 {milliseconds:.1f}밀리초')

벡터 100,000개, 154MB, 질문 하나의 검색에 26.3밀리초


인덱스는 253행 × 384열이고 크기는 0.389MB(388,608바이트)입니다. 253 × 384 × 4바이트(`float32`)를 계산한 값과 같습니다. 파일로 저장한 `index.npy`는 KB(1,024바이트)로 센 379KB이고, 같은 크기에 파일의 헤더가 조금 더해진 값입니다. 불러온 벡터는 저장하기 전 벡터와 완전히 같았습니다. 이 인덱스로 검색한 질문 20개의 recall@1은 85%, recall@3은 90%이고, 2일차에서 임베딩 검색으로 측정한 값과 같습니다.

`index.json`에 임베딩 모델의 이름을 함께 저장한 이유가 있습니다. 인덱스의 벡터는 그 모델이 만든 것이어서, 질문도 같은 모델로 임베딩해야 비교됩니다. 모델의 이름이 파일에 없으면 몇 달 뒤에 이 인덱스를 어느 모델로 검색해야 하는지 알 수 없습니다. 예제 1-3에서 다른 모델로 질문을 임베딩하면 어떻게 되는지 확인합니다.

문서가 수만 건이 되어도 이 방식으로 검색할 수 있습니다. 예제 1-1의 마지막 셀은 384차원 무작위 벡터 10만 개에서 질문 하나를 검색하는 시간을 측정합니다. 질문 벡터와 모든 문서 벡터의 내적을 한 번에 계산하므로, GPU가 없는 PC에서도 수십 밀리초 안에 끝납니다(값은 PC마다 다릅니다). 벡터가 100만 개이면 계산량은 10배이고, 벡터만으로 약 1.5GB(1,000,000 × 384 × 4바이트)의 메모리가 필요합니다. 병목은 검색 속도보다 운영 요건에서 먼저 생깁니다. 벡터가 모두 메모리에 올라가 있어야 하고, 여러 사람이 동시에 고치거나 접근 권한별로 검색 범위를 나누는 기능은 numpy 배열에 없습니다. 그런 기능이 필요하면 벡터 DB 제품을 씁니다.

검색만 빠르게 해 주는 라이브러리도 있습니다. FAISS(C++로 작성했고 Python 바인딩을 제공합니다)는 모든 벡터와 비교하는 정확한 검색과 가까운 후보만 찾는 근사 검색을 모두 지원합니다. hnswlib는 HNSW(Hierarchical Navigable Small World)라는 그래프 기반 근사 검색 알고리즘을 구현했습니다. 근사 검색은 가장 가까운 벡터를 가끔 놓치는 대신, 모든 벡터와 비교하는 검색보다 빠릅니다. 어느 쪽을 쓰든 인덱싱, 검색, 생성의 단계 구분은 같습니다.

> **보충**
>
> FAISS와 hnswlib의 설명은 각 저장소의 README에서 확인했습니다(2026-11-24 확인).

## 문서가 바뀔 때

> **요점**
>
> 문서가 개정되면 그 행의 벡터를 다시 계산해 교체하고, 새 문서는 행을 추가하고, 폐기한 문서는 행을 삭제합니다. 벡터와 문서 번호를 따로 고치면 어긋나므로 한 객체에 묶어 함께 고칩니다.

문서가 바뀌었다고 인덱스 전체를 다시 만들지는 않습니다. 바뀐 문서의 행만 고칩니다.

- **개정**: 그 문서의 본문으로 벡터를 다시 계산해 같은 행을 교체합니다.
- **신규**: 벡터를 계산해 맨 아래에 행을 추가합니다.
- **폐기**: 그 문서의 행을 삭제합니다.

이때 조심할 점이 있습니다. 벡터 배열과 문서 번호 목록은 같은 순서여야 하는데, 둘을 따로 고치면 한쪽만 고치고 끝내는 실수가 생깁니다. 코퍼스에는 `SOP-ETCH-012`의 구판(v1.2)과 개정판(v2.0)이 모두 있습니다. 구판 문서의 벡터만 지우고 문서 번호 목록은 그대로 두면 어떻게 되는지 봅니다.

**예제 1-2. 문서의 개정, 추가, 폐기를 행 단위로 반영하기**

In [ ]:
# 예제 1-2 (벡터만 지우고 문서 번호를 지우지 않으면)
old = names.index('SOP-ETCH-012_v1.2')                     # 구판 문서의 행
broken_vectors = np.delete(vectors, old, axis=0)           # 벡터만 지웁니다. names는 그대로입니다
print(f'벡터 {len(broken_vectors)}행, 문서 번호 {len(names)}개')

question = 'He 배면압 설정 기준이 어떻게 되나요'
scores = broken_vectors @ embed.query(question)
for row in np.argsort(-scores, kind='stable')[:3]:
    # 지운 행 뒤의 벡터는 한 칸씩 당겨졌으므로, 그 행의 실제 문서는 names에서 한 칸 뒤의 이름입니다
    print(f'{names[row]}  (이 행의 벡터는 실제로 {names[row + (row >= old)]}의 것입니다)')

벡터 252행, 문서 번호 253개
SOP-ETCH-012_v1.2  (이 행의 벡터는 실제로 SOP-ETCH-012_v2.0의 것입니다)
SOP-CMP-023  (이 행의 벡터는 실제로 SOP-CMP-023의 것입니다)
MTG-2025-W02  (이 행의 벡터는 실제로 MTG-2025-W02의 것입니다)


벡터, 문서 번호, 본문을 한 객체에 두고 항상 같이 고치는 클래스를 만듭니다. 벡터를 계산하는 일은 클래스 밖에서 하고, 클래스는 계산된 벡터를 받아 저장하고 고치고 찾는 일만 합니다.

In [ ]:
# 예제 1-2 (VectorIndex 클래스)
class VectorIndex:
    """벡터, 문서 번호, 본문을 한 객체에 두고 항상 같은 순서로 고칩니다."""

    def __init__(self, names, docs, vectors, model):
        if not len(names) == len(docs) == len(vectors):
            raise ValueError(f'개수가 다릅니다: 문서 번호 {len(names)}, 본문 {len(docs)}, 벡터 {len(vectors)}')
        self.names, self.docs, self.model = list(names), list(docs), model
        self.vectors = np.array(vectors)

    def __len__(self):
        return len(self.names)

    def search(self, query_vector, k=3):
        """질문 벡터와 가까운 상위 k건의 (문서 번호, 점수)를 돌려줍니다."""
        scores = self.vectors @ query_vector
        order = np.argsort(-scores, kind='stable')[:k]
        return [(self.names[row], round(float(scores[row]), 3)) for row in order]

    def rank_of(self, query_vector, gold):
        """정답 문서 gold가 몇 위인지 돌려줍니다. 1위가 1입니다."""
        order = list(np.argsort(-(self.vectors @ query_vector), kind='stable'))
        return order.index(self.names.index(gold)) + 1

    def replace(self, name, doc, vector):
        """개정: 같은 행의 본문과 벡터를 바꿉니다."""
        row = self.names.index(name)
        self.docs[row], self.vectors[row] = doc, vector

    def add(self, name, doc, vector):
        """신규: 맨 아래에 행을 추가합니다."""
        if name in self.names:
            raise ValueError(f'{name}은 이미 있습니다. 고치려면 replace()를 호출합니다')
        self.names.append(name)
        self.docs.append(doc)
        self.vectors = np.vstack([self.vectors, vector])

    def remove(self, name):
        """폐기: 그 문서의 행을 지웁니다."""
        row = self.names.index(name)
        del self.names[row], self.docs[row]
        self.vectors = np.delete(self.vectors, row, axis=0)

    def save(self, folder):
        np.save(f'{folder}/index.npy', self.vectors)
        with open(f'{folder}/index.json', 'w', encoding='utf-8') as file:
            json.dump({'model': self.model, 'names': self.names, 'docs': self.docs}, file, ensure_ascii=False)

    @classmethod
    def load(cls, folder, model=None):
        """저장한 인덱스를 불러옵니다. model을 주면 저장된 모델 이름과 같은지 확인하고, 다르면 ValueError를 냅니다."""
        with open(f'{folder}/index.json', encoding='utf-8') as file:
            saved = json.load(file)
        if model is not None and saved['model'] != model:
            raise ValueError(f'이 인덱스는 {saved["model"]}로 만들었는데 지금 모델은 {model}입니다')
        return cls(saved['names'], saved['docs'], np.load(f'{folder}/index.npy'), saved['model'])

예제 1-1에서 저장한 파일을 불러옵니다. 문서 한 건을 개정합니다. 질문은 두 가지입니다. 하나는 개정 문장의 낱말로 묻고, 다른 하나는 같은 내용을 다른 말로 묻습니다. 개정을 반영하기 전과 후에, 두 질문에서 이 문서의 순위를 비교합니다. 반영한 결과는 `out/updated`에 따로 저장해 예제 1-1의 파일을 그대로 둡니다. 추가와 폐기 셀은 개정 셀이 고친 `index`를 이어서 고치므로, 다시 실행하려면 개정 셀부터 순서대로 실행합니다.

In [ ]:
# 예제 1-2 (개정을 반영합니다)
index = VectorIndex.load(OUT, model=embed.model_name())      # 저장된 모델 이름이 지금 모델과 다르면 ValueError가 발생합니다
target = 'SOP-ETCH-007_v2.1'
asked = '온도 초과 판정을 몇 분으로 단축했나요'                 # 개정 문장과 같은 낱말을 씁니다
paraphrased = 'ESC 온도가 높을 때 챔버가 멈추기까지 얼마나 걸리나요'   # 같은 내용을 다른 말로 묻습니다

for label, question in [('개정 문장의 낱말로 묻기', asked), ('다른 말로 묻기', paraphrased)]:
    print(f'{label}, 반영 전 {index.rank_of(embed.query(question), target)}위')

revised = index.docs[index.names.index(target)] + '\n\n## 2026-10 개정\n- 상한 초과 판정 시간을 5분에서 3분으로 단축합니다.'
index.replace(target, revised, embed.passages([revised])[0])      # 이 행의 벡터 하나만 다시 계산합니다

for label, question in [('개정 문장의 낱말로 묻기', asked), ('다른 말로 묻기', paraphrased)]:
    print(f'{label}, 반영 후 {index.rank_of(embed.query(question), target)}위')
print('행 수', len(index))

개정 문장의 낱말로 묻기, 반영 전 4위
다른 말로 묻기, 반영 전 1위
개정 문장의 낱말로 묻기, 반영 후 1위
다른 말로 묻기, 반영 후 1위
행 수 253


신규 문서를 추가하고 구판 문서를 폐기합니다. 행 수가 그대로여도 벡터, 문서 번호, 본문의 개수가 모두 같은지 확인합니다.

In [ ]:
# 예제 1-2 (추가와 폐기를 반영합니다)
notice = ('# NOTI-099 사내 공지 — ESC 온도 기준 개정 안내\n\n'
          '- 게시일: 2026-10-01 · 게시 부서: 설비기술팀\n\n'
          'SOP-ETCH-007 의 상한 초과 판정 시간이 5분에서 3분으로 바뀌었습니다.')
index.add('NOTI-099', notice, embed.passages([notice])[0])
index.remove('SOP-ETCH-012_v1.2')
print(f'행 수 {len(index)}, 벡터 {index.vectors.shape}, 문서 번호 {len(index.names)}, 본문 {len(index.docs)}')
print('개정 문장의 낱말로 묻기 상위 3건', index.search(embed.query(asked), 3))

UPDATED = f'{OUT}/updated'                    # 예제 1-1의 파일은 그대로 두고 반영한 결과를 따로 저장합니다
os.makedirs(UPDATED, exist_ok=True)
index.save(UPDATED)
print('저장한 파일:', sorted(os.listdir(UPDATED)))

행 수 253, 벡터 (253, 384), 문서 번호 253, 본문 253
개정 문장의 낱말로 묻기 상위 3건 [('NOTI-099', 0.881), ('SOP-ETCH-007_v2.1', 0.839), ('QIR-2026-0314', 0.831)]
저장한 파일: ['index.json', 'index.npy']


벡터만 지우고 문서 번호 목록을 그대로 둔 첫 번째 코드에서는 `SOP-ETCH-012_v1.2`가 1위로 나왔습니다. 그런데 그 행의 벡터는 지운 구판이 아니라 그 바로 뒤에 있던 `SOP-ETCH-012_v2.0`의 것입니다. 지운 행 뒤의 모든 행이 한 칸씩 앞으로 당겨졌는데 문서 번호는 그대로여서, 이름과 벡터가 서로 다른 문서를 가리킵니다. 구판을 지웠는데 검색 결과에는 구판의 이름이 남아 있습니다. 오류도 나지 않으므로 응답이 이상할 때까지 알 수 없습니다. `VectorIndex`는 세 가지를 한 객체에 두고 `replace`, `add`, `remove`가 항상 함께 고치므로 이런 어긋남이 생기지 않습니다. `__init__`은 세 가지의 개수가 다르면 `ValueError`를 냅니다.

개정을 반영한 결과를 봅니다. 개정 문장과 같은 낱말로 물으면 `SOP-ETCH-007_v2.1`의 순위가 4위에서 1위로 올랐습니다. 이 문서의 벡터 한 행만 다시 계산했습니다. 같은 내용을 다른 말로 물은 질문은 반영하기 전에도 1위였습니다. 개정 전의 문서도 같은 주제(판정 시간)를 이미 적고 있어서, 5분이 3분으로 바뀐 것은 순위로 드러나지 않았습니다. 순위가 올랐다는 것은 새 벡터가 인덱스에 반영됐다는 증거이고, 값(3분)이 맞게 들어갔는지는 문서 본문을 열어 확인합니다. 따라서 개정이 반영됐는지 확인할 때는 개정에서 새로 들어간 낱말(이 예제에서는 "단축")을 넣은 질문으로 순위가 오르는지 봅니다. 순위가 오르지 않으면 그 행을 다시 계산했는지 의심합니다.

추가와 폐기를 반영한 뒤에도 행 수는 253행 그대로입니다. 한 건을 추가하고 한 건을 지웠기 때문입니다. 행 수만으로는 반영이 맞는지 알 수 없으므로 벡터, 문서 번호, 본문의 개수가 모두 같은지 함께 확인합니다. 추가한 개정 안내 공지(`NOTI-099`)는 코사인 유사도가 0.881로, 원문 SOP(`SOP-ETCH-007_v2.1`)의 0.839보다 높게 나왔습니다. 같은 내용을 적은 두 문서가 상위 3건에 함께 들어옵니다. 공지가 원문보다 위에 오면 응답의 출처가 원문 SOP가 아니라 공지가 될 수 있고, 같은 내용이 top-k의 자리를 두 개 차지합니다.

폐기한 문서는 인덱스에서 반드시 지웁니다. 구판 문서가 인덱스에 남아 있으면 구판의 값이 응답에 섞입니다. 파트 3에서 이 문제를 직접 확인합니다.

어떤 문서가 바뀌었는지 알아내는 방법(파일의 수정 시각이나 내용 비교)은 이 교재에서 다루지 않습니다(원본 강의 계획에서는 5일차 주제입니다).

## 임베딩 모델을 고르는 기준

> **요점**
>
> 임베딩 모델은 한 번 정하면 바꾸기 비쌉니다. 한국어로 학습했는지, 최대 입력 길이, 차원, 실행 위치를 보고 후보를 고른 뒤 사내 질문 세트로 측정해 정합니다. 모델을 바꾸면 인덱스 전체를 다시 만들어야 합니다.

임베딩 모델은 글을 입력받아 벡터를 출력하고, 글을 생성하지 않습니다(2일차 파트 3). 그래서 고르는 기준도 생성 모델과 다릅니다. 응답의 품질이 아니라 의미가 비슷한 글을 가까운 벡터로 바꾸는지를 봅니다.

고르는 기준은 네 가지입니다.

- **학습한 언어**: 한국어 문장으로 학습한 모델이어야 합니다. 여러 언어로 학습한 모델(multilingual)이나 한국어 전용 모델이 해당합니다. 영어만으로 학습한 모델이 한국어 문서를 얼마나 찾는지는 예제 1-3에서 측정합니다.
- **최대 입력 길이**: 한 번에 읽을 수 있는 토큰 수의 상한입니다(최대 입력 길이). 이보다 긴 글은 뒷부분이 잘려서 벡터에 반영되지 않습니다. 생성 모델의 컨텍스트 윈도우와 별개로 임베딩 모델마다 정해져 있습니다. 문서를 청크로 나누는 이유가 여기에 있고, 파트 2에서 확인합니다.
- **차원**: 벡터의 숫자 개수입니다. 차원이 클수록 벡터 인덱스의 저장 크기와 검색 계산량이 늘어납니다. 이 교재의 모델은 384차원입니다.
- **실행 위치**: 모델을 PC에서 실행하는지, API로 호출하는지입니다. 회사 밖의 서버에 있는 API로 호출하면 문서의 글이 그 서버로 갑니다. 사내 문서를 밖으로 보낼 수 없다면 PC에서 실행하는 모델이나 사내 서버의 모델을 고릅니다. 이 교재의 임베딩 모델은 PC에서 실행하므로 문서가 PC 밖으로 나가지 않습니다. 생성 모델의 API에는 검색 결과로 고른 문서만 보냅니다.

공개 순위표는 후보를 고르는 데 씁니다. MTEB(Massive Text Embedding Benchmark)가 대표적이고, 임베딩 모델을 여러 언어와 과제에서 평가한 점수를 보여 줍니다. 이 점수는 공개 데이터셋에서 측정한 값입니다. 사내 문서에서 현장 말투, 약어, 문서 번호가 섞인 질문을 얼마나 찾는지는 알려 주지 않습니다. 후보 두세 개를 정한 뒤 사내 문서와 질문 세트로 recall@k를 측정해 최종으로 고릅니다.

> **보충**
>
> MTEB가 임베딩 모델을 여러 언어와 과제에서 평가하는 벤치마크라는 설명은 MTEB 저장소의 README에서 확인했습니다(2026-11-24 확인).

한국어로 학습한 모델과 영어 전용 모델을 같은 질문 세트로 비교합니다. 영어 전용 모델로는 `sentence-transformers/all-MiniLM-L6-v2`를 씁니다. 이 모델도 384차원이어서, 차원을 같게 두고 학습한 언어의 효과를 비교할 수 있습니다. 비교 대상인 이 교재의 모델은 `intfloat/multilingual-e5-small`이고 여러 언어로 학습했습니다. 이 모델을 다국어 모델이라고 부릅니다. 영어 전용 모델을 내려받지 못하면 예제는 "건너뜁니다"를 출력하고 끝납니다.

**예제 1-3. 임베딩 모델을 바꾸면**

In [ ]:
# 예제 1-3
from sentence_transformers import SentenceTransformer

try:
    english = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
except Exception as error:
    english = None
    print('영어 전용 모델을 내려받지 못해 이 예제를 건너뜁니다.', type(error).__name__)

multilingual = embed.load()                 # 이 교재의 임베딩 모델
if english is not None:
    print(f'다국어 모델 {multilingual.get_embedding_dimension()}차원, 최대 입력 길이 {multilingual.max_seq_length}토큰')
    print(f'영어 전용 모델 {english.get_embedding_dimension()}차원, 최대 입력 길이 {english.max_seq_length}토큰')

다국어 모델 384차원, 최대 입력 길이 512토큰
영어 전용 모델 384차원, 최대 입력 길이 256토큰


영어 전용 모델로 문서 253건의 인덱스를 새로 만들고, 같은 질문 20개의 정답 순위를 두 모델에서 비교합니다.

In [ ]:
# 예제 1-3 (같은 질문으로 두 모델을 측정합니다)
def measure(label, rank_list):
    print(f'{label}: recall@1 {sum(rank <= 1 for rank in rank_list) / len(rank_list):.0%}, '
          f'recall@3 {sum(rank <= 3 for rank in rank_list) / len(rank_list):.0%}, 최저 순위 {max(rank_list)}위')


def ranks_with(doc_vectors, query_vectors):
    result = []
    for query_vector, gold in zip(query_vectors, questions['정답']):
        order = list(np.argsort(-(doc_vectors @ query_vector), kind='stable'))
        result.append(order.index(names.index(gold)) + 1)
    return result


if english is not None:
    english_docs = english.encode(docs, normalize_embeddings=True, batch_size=32)
    english_queries = english.encode(list(questions['질문']), normalize_embeddings=True)
    multilingual_queries = [embed.query(question) for question in questions['질문']]

    measure('다국어 모델', ranks_with(vectors, multilingual_queries))
    measure('영어 전용 모델', ranks_with(english_docs, english_queries))

다국어 모델: recall@1 85%, recall@3 90%, 최저 순위 9위
영어 전용 모델: recall@1 20%, recall@3 30%, 최저 순위 211위


두 모델의 토크나이저가 같은 문서를 몇 토큰으로 나누는지 셉니다. 최대 입력 길이를 넘는 문서는 뒷부분이 잘립니다.

In [ ]:
# 예제 1-3 (두 모델의 토크나이저가 문서를 몇 토큰으로 나누는지)
if english is not None:
    for label, model in [('다국어 모델', multilingual), ('영어 전용 모델', english)]:
        counts = [len(model.tokenizer(doc, verbose=False)['input_ids']) for doc in docs]   # verbose=False: 길이 초과 경고를 끕니다
        over = sum(count > model.max_seq_length for count in counts)
        print(f'{label}: "챔버" → {model.tokenizer.tokenize("챔버")}, 문서 토큰 수의 중앙값 {int(np.median(counts))}, '
              f'최대 입력 길이({model.max_seq_length}토큰)를 넘는 문서 {len(docs)}건 중 {over}건')

다국어 모델: "챔버" → ['▁', '챔', '버'], 문서 토큰 수의 중앙값 139, 최대 입력 길이(512토큰)를 넘는 문서 253건 중 0건
영어 전용 모델: "챔버" → ['ᄎ', '##ᅢ', '##ᆷ', '##ᄇ', '##ᅥ'], 문서 토큰 수의 중앙값 314, 최대 입력 길이(256토큰)를 넘는 문서 253건 중 223건


문서는 다국어 모델의 벡터로 두고 질문만 영어 전용 모델로 임베딩하면 어떻게 되는지도 봅니다. 두 모델이 모두 384차원이어서 오류 없이 계산됩니다.

In [ ]:
# 예제 1-3 (문서와 질문의 모델이 다르면)
if english is not None:
    mixed = ranks_with(vectors, english_queries)           # 문서는 다국어 모델, 질문은 영어 전용 모델
    measure('문서와 질문의 모델이 다름', mixed)
    print(f'순위의 중앙값 {int(np.median(mixed))}위 (문서 253건을 무작위로 정렬하면 중앙값이 127위 근처)')

문서와 질문의 모델이 다름: recall@1 5%, recall@3 5%, 최저 순위 233위
순위의 중앙값 73위 (문서 253건을 무작위로 정렬하면 중앙값이 127위 근처)


같은 질문 20개에서 영어 전용 모델은 recall@1이 20%, recall@3이 30%이고 정답 문서가 가장 낮게 나온 순위는 211위입니다. 다국어 모델은 recall@1이 85%, recall@3이 90%이고 가장 낮은 순위가 9위입니다. 두 모델의 차원은 같으므로 차원이 이 차이의 원인은 아닙니다.

차이는 학습한 언어에서 나옵니다. 영어 전용 모델의 토크나이저는 한글을 자모 단위로 쪼갭니다("챔버" → `ᄎ`, `##ᅢ`, `##ᆷ`, `##ᄇ`, `##ᅥ`). 어휘가 영어 위주여서 한국어 낱말이 토큰으로 들어 있지 않기 때문입니다. 그래서 같은 문서도 토큰 수가 다국어 모델의 2배가 넘고, 문서 253건 중 223건이 최대 입력 길이(256토큰)를 넘어 뒷부분이 잘렸습니다. 다국어 모델에서는 최대 입력 길이(512토큰)를 넘는 문서가 없었습니다. 자모로 쪼갠 토큰에서는 낱말의 뜻을 얻기 어렵고, 잘린 뒷부분은 벡터에 반영되지도 않습니다. 학습한 언어가 어휘와 잘림을 함께 바꾸므로, 한국어 문서의 임베딩 모델을 고를 때는 차원이나 속도보다 한국어로 학습했는지를 먼저 봅니다.

문서는 다국어 모델의 벡터로 두고 질문만 영어 전용 모델로 임베딩하면 recall@1과 recall@3이 모두 5%(질문 20개 중 1개)이고, 정답 문서의 순위는 중앙값이 73위, 가장 낮은 것이 233위입니다. 중앙값 73위는 무작위로 정렬했을 때(127위 근처)보다 낫지만, recall@3이 5%여서 검색에 쓸 수 없습니다. 차원이 같아서 계산은 오류 없이 되지만, 두 모델이 같은 위치에 놓는 숫자는 서로 다른 의미를 가집니다.

그래서 모델을 바꿀 때는 문서의 벡터를 새 모델로 모두 다시 계산하고, 질문도 새 모델로 임베딩해야 합니다. 인덱스에 모델의 이름을 함께 저장하는 것은 이 불일치를 알아챌 수 있게 하기 위해서입니다. `VectorIndex.load()`에 `model`을 주면 저장된 이름과 지금 쓰는 모델의 이름을 비교해, 다르면 `ValueError`를 냅니다(예제 1-2). 이 비교로 다른 모델로 검색하는 실수를 막을 수 있습니다.

모델을 바꾸면 문서 전체를 다시 임베딩해야 하므로, 처음 고를 때 사내 질문 세트로 충분히 측정합니다.

## RAG가 맞는 과제

> **요점**
>
> 문서가 자주 바뀌고 답마다 출처가 필요한 지식은 RAG로 다룹니다. 문서가 한두 건이고 한 번 쓰고 끝나는 일은 프롬프트에 넣으면 됩니다. 데이터에서 계산해야 하는 질문은 RAG로 답할 수 없습니다.

RAG를 만드는 데는 인덱싱, 검색, 생성을 모두 운영하는 비용이 듭니다. 과제를 받으면 RAG를 만들기 전에 다음 네 가지를 순서대로 묻습니다.

1. 답이 문서에 적혀 있습니까? 아니면 RAG가 아닙니다. 답이 공정 데이터나 센서 기록에 있고 계산이 필요한 질문은 문서를 검색해도 답이 나오지 않습니다. 데이터 분석 과제입니다.
2. 문서 전체가 컨텍스트 윈도우에 들어갑니까? 들어가지 않으면 문서를 프롬프트에 모두 넣을 수 없으므로 RAG를 씁니다.
3. 문서가 자주 바뀌거나 답마다 출처를 밝혀야 합니까? 그렇다면 RAG를 씁니다. 문서가 개정되면 해당 행만 갱신하고, 응답에 문서 번호를 붙일 수 있습니다.
4. 문서가 한두 건이고 개정과 출처가 필요 없습니까? 그렇다면 문서를 프롬프트에 넣으면 됩니다. 인덱스를 만들 이유가 없습니다.

3번과 4번 어느 쪽도 아니면(예: 문서가 컨텍스트 윈도우에 들어가지만 수백 건이고, 개정이 드물며 출처도 필요 없을 때) 문서 분량과 질문 수로 요금을 비교해 정합니다.

문서가 많다는 것이 RAG를 쓰는 이유의 전부는 아닙니다. 2일차에서 문서 253건(70,937자)을 모두 프롬프트에 넣어도 모델이 답했습니다. 문서가 많아도 컨텍스트 윈도우에 들어가는 분량이면 모두 넣을 수 있습니다. 그래도 RAG를 쓰는 이유는 다음과 같습니다.

- 문서를 모두 넣으면 질문마다 문서 전체의 입력 토큰 요금이 듭니다.
- 문서가 개정될 때마다 프롬프트에 넣을 문서를 사람이 고르고 바꿔야 합니다. RAG는 인덱스에서 바뀐 행만 고칩니다.
- 검색 결과 몇 건만 넣으면 응답의 근거가 될 수 있는 문서가 몇 건으로 줄어, 사람이 응답과 원문을 대조하기 쉽습니다.

분량은 숫자로 확인합니다. 다음은 문서 유형별로 글자 수를 세고 토큰 수를 추정하는 코드입니다.

**예제 1-4. 문서 유형별 분량 세기**

In [ ]:
# 예제 1-4
import unicodedata

manifest = pd.read_csv('data/2일차/corpus_manifest.csv', encoding='utf-8-sig')
# 한글이 든 파일 이름은 macOS에서 만든 압축 파일 등에서 NFD로 저장되기도 하므로, 양쪽을 NFC로 맞춰 비교합니다
length_of = {unicodedata.normalize('NFC', name): len(doc) for name, doc in zip(names, docs)}
manifest['글자 수'] = [length_of[unicodedata.normalize('NFC', file[:-3])] for file in manifest['file']]

by_type = manifest.groupby('type')['글자 수'].agg(['count', 'sum']).rename(columns={'count': '건수', 'sum': '글자 수'})
TOKENS_PER_CHAR = 67939 / 70937           # 2일차 예제 3-1에서 Anthropic 소형 모델에 문서 253건을 보내 측정한 값(입력 토큰 / 글자 수, 질문과 지시문 포함)
by_type['토큰(추정)'] = (by_type['글자 수'] * TOKENS_PER_CHAR).round().astype(int)
by_type.loc['전체'] = by_type.sum()
print(by_type.sort_values('글자 수', ascending=False).to_string())
print()
print(f"전체를 질문마다 넣으면 하루 1,000번 질문에 입력 토큰 {int(by_type.loc['전체', '토큰(추정)']) * 1000:,}개")
print(f"검색 결과 4건(문서 한 건 약 280자)만 넣으면 {round(4 * 280 * TOKENS_PER_CHAR) * 1000:,}개")

       건수   글자 수  토큰(추정)
type                    
전체    253  70937   67938
회의록    48  17523   16782
품질이슈   59  16064   15385
SOP    26  10486   10043
공정스펙   30  10301    9866
매뉴얼    25   4771    4569
공지     30   4275    4094
안전규정   20   4068    3896
교육자료   15   3449    3303

전체를 질문마다 넣으면 하루 1,000번 질문에 입력 토큰 67,938,000개
검색 결과 4건(문서 한 건 약 280자)만 넣으면 1,073,000개


문서 유형별로 분량이 다릅니다. 회의록이 17,523자로 가장 많고, 교육자료는 15건에 3,449자입니다. 전체는 70,937자이고 토큰으로는 약 68,000개입니다. 이 코퍼스의 문서는 한 건이 평균 280자 정도로 짧습니다. 토큰 수는 세지 않고 2일차 예제 3-1에서 측정한 비율로 추정했습니다. 이 비율은 Anthropic 소형 모델로 측정한 값이고, 질문과 지시문의 토큰도 들어 있습니다. 토크나이저가 다른 모델에서는 같은 글도 토큰 수가 다릅니다.

질문 한 번의 입력 토큰은 문서 전체를 넣으면 약 67,900개, 검색 결과 4건만 넣으면 약 1,070개로 약 63배입니다. 4건은 파트 3에서 쓰는 k=4에 맞춘 값입니다. 이 비율은 질문 수와 관계없이 질문마다 같습니다. 하루에 질문이 1,000번이면 입력 토큰의 차이는 하루 약 6,690만 개입니다.

> **보충**
>
> 같은 문서를 요청마다 프롬프트 앞부분에 그대로 넣는다면 프롬프트 캐싱(prompt caching)으로 입력 요금을 줄일 수 있습니다. Anthropic API는 캐시에서 읽은 입력 토큰을 기본 입력 단가의 0.1배(Claude Opus 5.5와 Sonnet 5.5는 0.05배)로, 캐시에 쓰는 입력 토큰을 1.25배로 계산합니다. 캐시는 기본으로 5분 동안 유지되고, 그 안에 다시 쓰면 수명이 다시 5분으로 늘어납니다. 그래서 입력 토큰 수는 그대로 약 63배여도 요금의 차이는 그보다 작아집니다. 사내 OpenAI 호환 서버가 프롬프트 캐싱을 지원하는지는 따로 확인합니다. Anthropic의 프롬프트 캐싱 문서에서 확인했습니다(2026-11-26 확인).

이 숫자로 업무 여섯 개를 판단합니다. 분량은 위 결과이고, 근거는 위 네 질문에 대한 답입니다.

| 업무 | 문서 분량 | 판단 | 근거 |
| --- | --- | --- | --- |
| 회의록 한 건 요약 | 한 건 | 프롬프트에 넣기 | 문서 한 건이 프롬프트에 들어가고 한 번 쓰고 끝납니다 |
| 보고 메일 문장 다듬기 | 초안 한 건 | 프롬프트에 넣기 | 찾을 문서가 없고 다듬을 초안만 붙이면 됩니다 |
| 사내 문서 253건에서 기준값 찾기 | 70,937자 | RAG | 문서가 개정되고, 기준값에는 출처 문서 번호가 필요합니다 |
| 품질 이슈 보고서 59건에서 비슷한 사례 찾기 | 16,064자 | RAG | 보고서가 계속 쌓이고, 사례마다 보고서 번호를 밝혀야 합니다. 분량만 보면 프롬프트에 들어가지만 질문마다 전체의 요금이 듭니다 |
| 신입 교육 자료 질의응답 | 15건, 3,449자 | 프롬프트에 넣기 또는 RAG | 분량은 한 번에 들어갑니다. 개정이 자주 있거나 답마다 근거 자료를 보여 줘야 하면 RAG를 고려하고, 개정이 드물고 질문이 적고 출처가 필요 없으면 프롬프트에 넣는 쪽이 간단합니다 |
| 이번 달 수율 하락 원인 | 문서가 아님 | RAG로 답할 수 없음 | 답이 공정 데이터에 있습니다. 데이터 분석 과제입니다 |

신입 교육 자료 질의응답은 이 표만으로 한쪽으로 정할 수 없습니다. 개정 빈도, 출처의 필요 여부, 질문 수를 알아야 정할 수 있습니다.

RAG, 프롬프트에 넣기, 파인튜닝은 서로 배타적이지 않고 함께 쓰기도 합니다. 모델이 모르는 사내 지식은 문서를 프롬프트에 넣어 제공합니다. 문서가 많거나 자주 바뀌면 RAG로 고른 문서를 넣고, 적으면 문서를 직접 넣습니다. 응답의 형식과 말투는 프롬프트의 지시와 예시로 대부분 맞출 수 있습니다. 지시가 길어져 매 요청의 비용이 크면 파인튜닝을 고려합니다. 같은 지시에도 형식이나 말투가 요청마다 달라질 때도 파인튜닝을 고려합니다. 파인튜닝은 6일차에서 다룹니다.

## 실습

실습은 `3일차_실습.ipynb`에서 합니다. 임베딩 모델과 API를 부르지 않는 테스트 셀로 함수를 검사하고, 실제 문서로 반영하는 셀에서만 임베딩 모델을 실행합니다. 풀이와 해설은 `3일차_실습_풀이.ipynb`에 있습니다.

**실습 1. 변경 목록을 인덱스에 반영하기**

변경 목록(개정 2건, 신규 2건, 폐기 5건)을 벡터 인덱스에 반영하는 `apply_changes()`를 직접 작성합니다. 변경 목록에 인덱스에 없는 문서가 들어 있으면 아무것도 고치지 않고 `ValueError`를 발생시켜야 합니다. 이어서 변경의 일부를 빠뜨린 두 경우의 검색 결과를 읽고, 현장에서 받는 응답이 어떻게 달라지는지 판단합니다.

완료 기준: 테스트 셀이 "실습 1 통과"를 출력합니다. 반영 후 인덱스가 250행이고 벡터, 문서 번호, 본문의 개수가 같습니다. 두 경우의 판단을 근거 숫자와 함께 적습니다.

## 핵심 정리

- RAG는 인덱싱, 검색, 생성의 세 단계입니다. 인덱싱은 문서가 바뀔 때만, 검색과 생성은 질문마다 실행합니다. 응답이 틀리면 검색 결과부터 확인합니다.
- 벡터 인덱스는 파일로 저장해 다시 씁니다. 벡터와 함께 문서 번호와 임베딩 모델의 이름을 저장합니다.
- 문서가 바뀌면 바뀐 행만 고칩니다. 개정은 교체, 신규는 추가, 폐기는 삭제입니다. 벡터와 문서 번호를 한 객체에 두고 함께 고쳐 어긋남을 막습니다.
- 갱신이 반영됐는지는 개정에서 새로 들어간 낱말을 넣은 질문의 순위로 확인합니다. 행 수만으로는 알 수 없고, 값이 맞는지는 본문을 열어 확인합니다.
- 임베딩 모델은 한국어로 학습했는지, 최대 입력 길이, 차원, 실행 위치를 보고 후보를 정하고, 사내 질문 세트로 측정해 고릅니다. 모델을 바꾸면 인덱스 전체를 다시 만듭니다.
- 문서가 컨텍스트 윈도우에 들어가지 않거나, 자주 바뀌고 출처가 필요하면 RAG를, 문서가 한두 건이고 한 번 쓰고 끝나면 프롬프트에 넣기를 씁니다. 답이 데이터에 있는 질문은 RAG로 답할 수 없습니다.

## 자가점검

**1.** RAG의 세 단계(인덱싱, 검색, 생성)의 실행 시점을 바르게 설명한 것은 무엇입니까?

- A. 인덱싱은 질문이 올 때마다, 검색과 생성은 문서가 바뀔 때만 실행한다
- B. 인덱싱은 문서가 바뀔 때만, 검색과 생성은 질문이 올 때마다 실행한다
- C. 인덱싱과 검색은 문서가 바뀔 때만, 생성은 질문이 올 때마다 실행한다

<details><summary>정답 보기</summary>

**B.** 인덱싱은 문서를 임베딩해 인덱스에 넣는 일이어서 문서가 바뀔 때만 합니다. 검색과 생성은 질문마다 합니다.

</details>

**2.** 운영 중인 벡터 인덱스에서 바뀐 행만 고치지 않고 전체를 다시 만들어야 하는 경우는 언제입니까?

- A. 문서 한 건이 개정됐을 때
- B. 새 문서가 여러 건 추가됐을 때
- C. 임베딩 모델을 다른 모델로 바꿨을 때
- D. 더 이상 쓰지 않는 문서를 폐기했을 때

<details><summary>정답 보기</summary>

**C.** 모델이 다르면 같은 문서도 다른 벡터가 되고, 두 모델의 벡터는 서로 비교할 수 없습니다(예제 1-3). 문서의 개정, 추가, 폐기는 해당 행만 고치면 됩니다.

</details>

**3.** RAG에서 임베딩 모델이 하는 일로 옳은 것은 무엇입니까?

- A. 질문을 받아 이어질 응답 문장을 생성한다
- B. 검색 결과를 읽고 응답에 출처를 붙인다
- C. 글을 벡터로 바꾼다. 검색 단계는 이 벡터끼리의 유사도로 문서를 찾는다

<details><summary>정답 보기</summary>

**C.** 유사도를 계산하는 것은 인덱스를 검색하는 코드입니다. 임베딩 모델은 글을 벡터로 바꾸는 데서 끝나고 글을 생성하지 않습니다. 응답을 생성하고 출처를 붙이는 것은 생성 모델의 일입니다.

</details>

**4.** 임베딩 모델 공개 순위표에서 1위인 모델을 골랐습니다. 이 선택에 대한 설명으로 옳은 것은 무엇입니까?

- A. 공개 데이터셋에서의 순위여서, 사내 문서와 현장 말투 질문에서의 검색 성능은 알려 주지 않는다
- B. 1위 모델은 어떤 언어와 문서에서도 검색 성능이 가장 높다
- C. 차원이 같으면 영어 전용 모델로도 한국어 문서를 비슷하게 검색할 수 있다

<details><summary>정답 보기</summary>

**A.** 순위표는 후보를 고르는 데 쓰고, 최종 선택은 사내 문서와 질문 세트로 recall@k를 측정해서 합니다. C는 예제 1-3에서 틀렸음을 확인했습니다. 차원은 같아도 영어 전용 모델은 한국어 문서를 거의 찾지 못했습니다.

</details>

**5.** 사내 문서로 만든 RAG만으로 답하기 어려운 질문은 무엇입니까?

- A. 설비 규격서에 적힌 허용 온도 범위를 묻는 질문
- B. 작년 품질 보고서 가운데 비슷한 불량 사례를 찾는 질문
- C. 오늘 생산 라인의 센서 기록에서 온도가 허용 범위를 넘은 횟수를 묻는 질문
- D. 신입 교육 자료에 적힌 보호구 착용 규정을 묻는 질문

<details><summary>정답 보기</summary>

**C.** 답이 문서가 아니라 센서 기록에 있고, 횟수를 세는 계산이 필요한 데이터 분석 과제입니다. 문서를 검색해도 답이 나오지 않습니다.

</details>

**6.** 구판 문서의 벡터만 지우고 문서 번호 목록을 그대로 두면 어떤 일이 생깁니까?

<details><summary>정답 보기</summary>

벡터가 한 행씩 앞으로 당겨져 문서 번호와 벡터가 서로 다른 문서를 가리킵니다. 오류가 나지 않으므로 검색 결과의 문서 번호가 틀렸다는 것을 알기 어렵습니다(예제 1-2). 벡터, 문서 번호, 본문을 한 객체에 두고 함께 고치면 막을 수 있습니다.

</details>

**7.** 문서 한 건을 개정하고 그 행의 벡터를 다시 계산했습니다. 반영이 검색에 나타났는지 확인하는 질문으로 알맞은 것은 무엇입니까?

- A. 개정 전에도 있던 내용을 묻는 질문
- B. 개정에서 새로 들어간 낱말을 넣은 질문
- C. 인덱스의 행 수를 세는 코드

<details><summary>정답 보기</summary>

**B.** 새로 들어간 낱말은 반영하기 전의 벡터에는 없었으므로 순위가 오르는지로 확인할 수 있습니다(예제 1-2). 개정 전에도 있던 내용을 묻는 질문은 반영하기 전에도 1위일 수 있고, 행 수는 교체해도 변하지 않습니다.

</details>

**8.** 신입 교육 자료 15건(3,449자)으로 질의응답을 만들려고 합니다. 질문은 하루에 열 번 정도이고 자료는 1년에 한 번 개정되며, 답마다 근거를 보여 줄 필요는 없습니다. RAG와 프롬프트에 넣기 가운데 어느 쪽이 간단합니까? 그 판단이 바뀌는 조건은 무엇입니까?

<details><summary>정답 보기</summary>

자료 전체가 프롬프트에 한 번에 들어가는 분량이고 질문이 적으므로 프롬프트에 넣는 쪽이 간단합니다. 자료가 크게 늘거나, 개정이 잦거나, 답마다 근거 자료를 보여 줘야 하면 RAG를 고려합니다.

</details>